# HPD 1 — Entregable: Pipeline de embeddings y búsqueda semántica

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd1-entregable.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 %
>
> **Plazo:** por determinar.
>
> **Entrega:** Notebook `.ipynb` ejecutado con todas las celdas
> completas. Cada ejercicio especifica qué variable debe contener el
> resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto (`eval1_estrategia`, `eval2_recall`, `eval3_busqueda`,
> `eval4_justificacion`). El script de corrección ejecutará tu notebook
> e inspeccionará esas variables.

## Setup

> **Tiempo de ejecución en Colab**
>
> Codificar los 5551 chunks del ejercicio 2 tarda unos **6-7 minutos**
> en una CPU de Colab y unos **9 segundos** con la GPU T4 gratuita
> (ambos tiempos medidos). Antes de empezar, activa la GPU en *Entorno
> de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*. Colab
> no garantiza GPU libre en todo momento; si no te la da, el cuaderno
> funciona igual con CPU, solo que esa celda tarda más. Los resultados
> (recall@k) son los mismos con CPU y con GPU.

In [1]:
!pip install -q chromadb==1.5.9 sentence-transformers==5.5.1 fpdf2==2.8.7 matplotlib "pandas<3" langchain==1.3.2 langchain-community==0.3.31 langchain-text-splitters==1.1.2 pypdf==6.12.2 python-dotenv==1.2.2 datasets==4.8.5 langchain-core==1.4.0 langgraph==1.2.2 openai==2.38.0

In [2]:
import os, tempfile, shutil
"""
Setup compartido para la API key de Llamus en todos los notebooks.
Inyectado en cada .qmd/.ipynb vía scripts/sync_setup_keys.py (pre-render).

Orden de resolución:
  1. .env local (python-dotenv)
  2. os.environ ya configurado externamente
  3. Google Colab userdata: secret "LLAMUS_API_KEY" (preferente) o "LLM_API_KEY"

Las celdas posteriores pueden leer la clave con os.environ.get("LLM_API_KEY").
"""

import os

def _load_llm_api_key():
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    if not os.getenv("LLM_API_KEY"):
        try:
            from google.colab import userdata
            for name in ("LLAMUS_API_KEY", "LLM_API_KEY"):
                try:
                    value = userdata.get(name)
                    if value:
                        os.environ["LLM_API_KEY"] = value
                        break
                except Exception:
                    continue
        except ImportError:
            pass

_load_llm_api_key()

if os.getenv("LLM_API_KEY"):
    print("✓ LLM_API_KEY cargada.")
else:
    print("⚠️  LLM_API_KEY no encontrada.")
    print("    • Local: crea un archivo .env con LLM_API_KEY=tu_clave")
    print("    • Colab: añade un Secret 🔑 con nombre 'LLAMUS_API_KEY'")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fpdf import FPDF
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 100
print("Librerías cargadas correctamente.")

✓ LLM_API_KEY cargada.
Librerías cargadas correctamente.

In [3]:
# ------------------------------------------------------------------ Parámetros compartidos
MODEL_NAME = "intfloat/e5-small-v2"
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

BASE_DIR = tempfile.mkdtemp()
CORPUS_DIR = os.path.join(BASE_DIR, "corpus")
os.makedirs(CORPUS_DIR, exist_ok=True)
print(f"Directorio temporal: {BASE_DIR}")

# ------------------------------------------------------------------ Modelo de embeddings
model = SentenceTransformer(MODEL_NAME)
print(f"Modelo cargado: {MODEL_NAME} ({model.get_embedding_dimension()} dimensiones)")

# ------------------------------------------------------------------ Chroma
client = chromadb.EphemeralClient()

Directorio temporal: /var/folders/5q/dq_s1n552d37rqhcc02ppsn00000gn/T/tmpc4b7l5ol

Modelo cargado: intfloat/e5-small-v2 (384 dimensiones)

In [4]:
class PDF(FPDF):
    pass

def crear_pdf(nombre: str, titulo: str, contenido: str) -> None:
    pdf = PDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    titulo_safe = titulo.replace("—", "-").replace("€", "EUR").replace("…", "...")
    pdf.cell(0, 10, titulo_safe, new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)
    pdf.set_font("Helvetica", "", 11)
    contenido_safe = contenido.replace("—", "-").replace("€", "EUR").replace("…", "...")
    pdf.multi_cell(0, 6, contenido_safe)
    pdf.output(os.path.join(CORPUS_DIR, nombre))
    print(f"  ✅ {nombre}")

print("Helper PDF listo.")

Helper PDF listo.

## Ejercicios — contexto financiero

**Contexto:** Trabajas como Data Scientist en una consultora financiera.
Tu cliente te pide un sistema de búsqueda semántica sobre documentación
financiera real del benchmark FIQA (Financial Question Answering), un
benchmark del ecosistema BEIR. El sistema debe permitir a los analistas
buscar información por concepto, no por palabra clave.

> **Anotaciones incompletas**
>
> Las relevancias de FiQA son **incompletas**: un documento que responde
> bien a una consulta pero que no aparece en el ground truth cuenta como
> **fallo** en recall@k. Ten esta limitación presente al interpretar las
> métricas.

In [5]:
import random

# ------------------------------------------------------------------ Muestreo FiQA (estilo BEIR)
N_QUERIES = 50       # consultas de evaluación (cada una con al menos 1 documento relevante)
N_DOCS_TOTAL = 1500  # tamaño aproximado del corpus: relevantes + distractores aleatorios
SEED = 42            # semilla fija: el mismo muestreo en cada ejecución

rng = random.Random(SEED)

# Qrels del split test (BeIR/fiqa-qrels tiene train, validation y test)
fiqa_qrels = load_dataset("BeIR/fiqa-qrels", split="test")
relevantes_por_query = {}
for row in fiqa_qrels:
    if row["score"] > 0:
        relevantes_por_query.setdefault(str(row["query-id"]), set()).add(str(row["corpus-id"]))
print(f"Consultas de test con al menos 1 documento relevante: {len(relevantes_por_query)}")

# 1) Consultas: N_QUERIES al azar (semilla fija)
qids_muestra = rng.sample(sorted(relevantes_por_query, key=int), N_QUERIES)

# 2) Documentos relevantes de esas consultas: todos
rel_ids = set()
for qid in qids_muestra:
    rel_ids |= relevantes_por_query[qid]

# 3) Distractores aleatorios hasta N_DOCS_TOTAL
fiqa_corpus = load_dataset("BeIR/fiqa", "corpus", split="corpus")
textos_corpus = dict(zip(fiqa_corpus["_id"], fiqa_corpus["text"]))
assert rel_ids <= textos_corpus.keys(), "Hay qrels que apuntan a documentos fuera del corpus"
candidatos = [i for i, txt in textos_corpus.items() if i not in rel_ids and txt.strip()]
distractores = rng.sample(candidatos, N_DOCS_TOTAL - len(rel_ids))
docs_ids = sorted(rel_ids | set(distractores), key=int)

# Consultas y ground truth restringidos a la muestra
fiqa_queries = load_dataset("BeIR/fiqa", "queries", split="queries")
qids_set = set(qids_muestra)
queries_map = {q["_id"]: q["text"] for q in fiqa_queries if q["_id"] in qids_set}
consultas_a = [(qid, queries_map[qid]) for qid in qids_muestra]    # lista de (qid, texto)
gt_a = {qid: relevantes_por_query[qid] for qid in qids_muestra}   # qid -> ids de documentos relevantes

docs_a = [
    Document(page_content=textos_corpus[i], metadata={"source": f"fiqa_{i}.txt", "fiqa_id": i})
    for i in docs_ids
]

pares = sum(len(v) for v in gt_a.values())
print(f"Consultas de evaluación: {len(consultas_a)}")
print(f"Documentos en corpus: {len(docs_a)} ({len(rel_ids)} relevantes + {len(distractores)} distractores)")
print(f"Pares consulta-documento relevante: {pares} (media {pares / len(gt_a):.2f} por consulta)")

Consultas de test con al menos 1 documento relevante: 648
Consultas de evaluación: 50
Documentos en corpus: 1500 (147 relevantes + 1353 distractores)
Pares consulta-documento relevante: 147 (media 2.94 por consulta)

#### Ejercicio 1 — Estrategia de chunking (2 ptos)

Describe qué estrategia de chunking eliges (chunk por párrafos
vs. tamaño fijo con solapamiento) y por qué es adecuada para documentos
financieros reales del benchmark FIQA. Justifica tu respuesta en
términos de:

- Coherencia semántica de los chunks
- Capacidad de recuperar información con consultas sobre magnitudes
  concretas (ej. “ratio de apalancamiento”, “ROE”, “morosidad”)
- Ventajas del solapamiento entre chunks en este dominio

Asigna tu respuesta a `eval1_estrategia`.

In [6]:
# TODO: sustituye el texto de la plantilla por tu justificación (ejercicio 1).
eval1_estrategia = """TODO: escribe aquí tu justificación (unas 100-200 palabras).

Preguntas que conviene responder: ¿qué pasa con un párrafo de FiQA que mezcla varios conceptos?
¿Qué ocurre con una cifra si el corte cae justo en medio de la frase? ¿Qué parámetros usas en el
ejercicio 2 y cómo afectan al tamaño de los chunks?"""
print("✅ eval1_estrategia asignada.")

✅ eval1_estrategia asignada.

#### Ejercicio 2 — Indexación y recall@k (2.5 ptos)

Genera los embeddings de los chunks de los documentos FIQA muestreados,
construye la colección en Chroma y evalúa **recall@k** sobre las 50
consultas usando el ground truth de las anotaciones reales del
benchmark.

Convención de la métrica: `k` es el número de chunks recuperados y el
recall se calcula **a nivel de documento**, es decir, qué fracción de
los documentos relevantes de cada consulta aparece entre los documentos
de los `k` primeros chunks. Promedia sobre las consultas. El resultado
queda en `eval2_recall`.

In [7]:
import time

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks_a = splitter.split_documents(docs_a)
print(f"Chunks generados: {len(chunks_a)}")

# HNSW con ef más alto: el índice aproximado da resultados estables y muy próximos a la búsqueda exacta
collection_a = client.get_or_create_collection(
    name="hpd1_finanzas",
    metadata={"hnsw:space": "cosine", "hnsw:construction_ef": 200, "hnsw:search_ef": 200, "hnsw:M": 32}
)

# Embeddings en lote (mucho más rápido que codificar chunk a chunk)
t0 = time.perf_counter()
ids_chunks = [f"fin_{i:05d}" for i in range(len(chunks_a))]
embeddings_chunks = model.encode(
    ["passage: " + c.page_content for c in chunks_a],
    normalize_embeddings=True, batch_size=32, show_progress_bar=False,
).tolist()
t_emb = time.perf_counter() - t0

for ini in range(0, len(chunks_a), 1000):
    fin = ini + 1000
    collection_a.add(
        ids=ids_chunks[ini:fin],
        embeddings=embeddings_chunks[ini:fin],
        documents=[c.page_content for c in chunks_a[ini:fin]],
        metadatas=[{"fiqa_id": c.metadata["fiqa_id"], "idx": ini + j}
                   for j, c in enumerate(chunks_a[ini:fin])],
    )
print(f"Indexación completada: {collection_a.count()} vectores ({t_emb:.1f} s generando embeddings)")

def buscar(collection, query, k=5):
    qe = model.encode("query: " + query, normalize_embeddings=True).tolist()
    return collection.query(query_embeddings=[qe], n_results=k)

def recall_at_k(metadatas, relevantes, k):
    """Recall a nivel de documento: fracción de documentos relevantes presentes
    entre los fiqa_id de los k primeros chunks recuperados."""
    docs_top_k = {m["fiqa_id"] for m in metadatas[:k]}
    return len(docs_top_k & relevantes) / len(relevantes)

KS = [1, 3, 5, 10]
busquedas = {qid: buscar(collection_a, q, k=max(KS)) for qid, q in consultas_a}

eval2_recall = {}
for k in KS:
    recalls = [recall_at_k(busquedas[qid]["metadatas"][0], gt_a[qid], k) for qid, _ in consultas_a]
    eval2_recall[f"recall@{k}"] = round(float(np.mean(recalls)), 4)

print(f"📊 recall@k sobre {len(consultas_a)} consultas FIQA (ground truth real, nivel documento):")
for k_str, v in eval2_recall.items():
    print(f"   {k_str}: {v:.4f}")

Chunks generados: 5551
Indexación completada: 5551 vectores (10.8 s generando embeddings)
📊 recall@k sobre 50 consultas FIQA (ground truth real, nivel documento):
   recall@1: 0.3822
   recall@3: 0.5998
   recall@5: 0.6667
   recall@10: 0.7584

#### Ejercicio 3 — Búsqueda semántica (1.5 ptos)

Ejecuta 3 consultas de búsqueda semántica sobre el corpus FIQA y muestra
los resultados (top-3 chunks con su distancia coseno). Las consultas son
preguntas reales del benchmark (las tres primeras de la muestra).

Asigna los resultados (lista de diccionarios con query, top-3 chunks y
distancias) a `eval3_busqueda`.

In [8]:
consultas_ej3 = [q for _, q in consultas_a[:3]]

eval3_busqueda = []
for q in consultas_ej3:
    res = buscar(collection_a, q, k=3)
    resultados = []
    for i in range(len(res["ids"][0])):
        resultados.append({
            "id": res["ids"][0][i],
            "distancia": round(res["distances"][0][i], 4),
            "texto": res["documents"][0][i][:200],
            "fuente": res["metadatas"][0][i].get("fiqa_id", ""),
        })
    eval3_busqueda.append({"query": q, "resultados": resultados})

for r in eval3_busqueda:
    print(f"\n🔍 {r['query']}")
    for res in r["resultados"]:
        print(f"   [{res['distancia']:.4f}] {res['texto'][:100]}...")
print("\n✅ eval3_busqueda asignada.")


🔍 Can I pay taxes using bill pay from my on-line checking account?
   [0.1280] . Read more on this topic at Tax topic 513...
   [0.1299] . But yes, you can do it, if you so wish. I suggest you talk to a licensed tax adviser (EA/CPA licen...
   [0.1305] I can't speak for the US, but I've completed direct tax payments via my online bank account (for bus...

🔍 Is there a reason to buy a 0% yield bond?
   [0.0965] No, there isn't. There are a number of reasons that institutions buy these bonds but as an individua...
   [0.1078] 0% bonds are desirable for some individuals. It depends on your situation. 0% bonds are usually sold...
   [0.1282] . But in my opinion, I would wonder - who cares? Yield is important to ensure you can hold the prope...

🔍 How to donate to charity that will make a difference?
   [0.1274] .  You can always do that later.  And try to donate anonymously if you can.  Donating often leads to...
   [0.1288] .  You can also rent them out if you want time to make a decisio

Función auxiliar — ya escrita, no es parte de tu solución. La usarás en
el ejercicio 4.

In [9]:
# ------------------------------------------------------------------ Función auxiliar (YA PROPORCIONADA)
def evaluar_modelo_alternativo(model_id, prefijo_consulta="", prefijo_doc="", emb_chunks=None):
    """Evalúa otro modelo de embeddings sobre EXACTAMENTE los mismos chunks, consultas y
    ground truth que los ejercicios 2 y 3, con búsqueda exacta por coseno (numpy, sin Chroma).

    Devuelve una tupla (medias, r10_por_consulta):
      - medias: {"recall@1": ..., "recall@3": ..., "recall@5": ..., "recall@10": ...}
                recall a nivel de documento, promediado sobre las consultas.
      - r10_por_consulta: lista con el recall@10 de cada consulta (mismo orden que consultas_a).

    Si ya tienes los embeddings de los chunks de ese modelo, pásalos en emb_chunks y no se
    vuelven a calcular (en una CPU, codificar los chunks es lo que más tarda).
    """
    modelo = SentenceTransformer(model_id)
    if emb_chunks is None:
        C = modelo.encode([prefijo_doc + c.page_content for c in chunks_a],
                          normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    else:
        C = np.asarray(emb_chunks)
    Q = modelo.encode([prefijo_consulta + q for _, q in consultas_a],
                      normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    fiqa_ids = [c.metadata["fiqa_id"] for c in chunks_a]

    S = Q @ C.T                                   # similitud coseno (vectores ya normalizados)
    top = np.argsort(-S, axis=1)[:, :10]          # los 10 chunks más parecidos, de mayor a menor

    KS = [1, 3, 5, 10]
    recalls = {k: [] for k in KS}
    for fila, (qid, _) in enumerate(consultas_a):
        rel = gt_a[qid]
        for k in KS:
            docs_top_k = {fiqa_ids[i] for i in top[fila, :k]}
            recalls[k].append(len(docs_top_k & rel) / len(rel))

    medias = {f"recall@{k}": round(float(np.mean(recalls[k])), 4) for k in KS}
    return medias, recalls[10]

print("✅ Función evaluar_modelo_alternativo lista.")

✅ Función evaluar_modelo_alternativo lista.

#### Ejercicio 4 — Justificación del modelo (1.5 ptos)

Tu cliente pregunta: **¿es `intfloat/e5-small-v2` una elección
defendible para este buscador, frente a un modelo entrenado para otra
tarea?**

Para responderlo, compara e5-small-v2 con
`sentence-transformers/paraphrase-MiniLM-L6-v2`. Es un modelo pequeño
que se entrenó para detectar **paráfrasis** (dos frases que dicen lo
mismo), no para encontrar documentos que respondan a una pregunta.
Compáralos **sobre la misma muestra** del ejercicio 2: los mismos
chunks, las mismas 50 consultas y el mismo ground truth. La función
`evaluar_modelo_alternativo` ya hace el trabajo de embeber y medir; tú
la llamas y analizas el resultado.

Tu justificación debe:

1.  Llamar a la función con el modelo de paráfrasis (TODO (a) de la
    celda) y presentar una tabla con `recall@1`, `recall@3`, `recall@5`
    y `recall@10` de los dos modelos, y el número de consultas en que
    gana cada uno (TODO (b)).
2.  Basarse en **tus cifras**. La nota de MTEB puede citarse como
    contexto, pero no sustituye a tu medida.
3.  Explicar con tus palabras por qué las tareas STS no bastan para
    decidir sobre un buscador (relaciona con la sección 2.6 del taller,
    «STS no es recuperación»).
4.  Incluir al menos una limitación de tu evaluación y cómo la
    mitigarías.
5.  Terminar con una conclusión prudente: ¿mantienes e5-small-v2 o
    cambias de modelo? ¿Qué medirías antes de decidir?

Asigna tu respuesta a `eval4_justificacion`.

> **Ojo con las conclusiones**
>
> Con esta muestra de 50 consultas, **e5-small-v2 no es demostrablemente
> mejor que all-MiniLM-L6-v2** en recall@10: la diferencia que observes
> es menor que el ruido de la muestra. En recall@1 sí puede aparecer una
> ventaja a favor de e5, pero depende de qué consultas caigan en la
> muestra. No afirmes que un modelo es mejor que otro sin evidencia que
> lo sostenga.
>
> Pregunta guía (opcional): si compararas e5-small-v2 con
> all-MiniLM-L6-v2 sobre esta misma muestra, ¿podrías afirmar cuál de
> los dos es mejor? ¿Qué te faltaría para poder hacerlo?

In [10]:
# --- Línea base: e5-small-v2 con el MISMO método exacto (ya escrita) -------------------------
# Reutiliza los embeddings del ejercicio 2 (embeddings_chunks): no hace falta codificar otra vez.
metricas_e5, r10_e5 = evaluar_modelo_alternativo("intfloat/e5-small-v2", "query: ", "passage: ",
                                                 emb_chunks=embeddings_chunks)
print("e5-small-v2: búsqueda HNSW del ejercicio 2 frente a búsqueda exacta de la función")
for k_str in eval2_recall:
    print(f"   {k_str}: HNSW = {eval2_recall[k_str]:.4f} | exacta = {metricas_e5[k_str]:.4f}")

# --- (a) TODO -------------------------------------------------------------------------------
# Llama a evaluar_modelo_alternativo con "sentence-transformers/paraphrase-MiniLM-L6-v2".
# Este modelo NO usa prefijos: no pases prefijo_consulta ni prefijo_doc.
# Guarda las medias en metricas_para y la lista de recall@10 por consulta en r10_para.
metricas_para, r10_para = None, None   # ← sustituye esta línea por tu llamada

# --- (b) TODO -------------------------------------------------------------------------------
# Imprime una tabla con recall@1, recall@3, recall@5 y recall@10 de los dos modelos
# (metricas_e5 y metricas_para), y el número de consultas en que gana cada uno comparando
# r10_e5 y r10_para consulta a consulta. Cuenta también los empates.
# Pista: sum(a > b for a, b in zip(lista_1, lista_2)) cuenta las consultas en que gana lista_1.

print("✅ Completa los TODO (a) y (b) y después eval4_justificacion.")

e5-small-v2: búsqueda HNSW del ejercicio 2 frente a búsqueda exacta de la función
   recall@1: HNSW = 0.3822 | exacta = 0.3822
   recall@3: HNSW = 0.5998 | exacta = 0.5998
   recall@5: HNSW = 0.6667 | exacta = 0.6667
   recall@10: HNSW = 0.7584 | exacta = 0.7584
✅ Completa los TODO (a) y (b) y después eval4_justificacion.

In [11]:
# TODO: sustituye el texto de la plantilla por tu justificación (ejercicio 4).
eval4_justificacion = """TODO: escribe aquí la justificación (unas 150-250 palabras). Responde a estos puntos:

1. Cifras: ¿qué modelo tiene mejor recall@10 en tu tabla? ¿Y mejor recall@1? ¿En cuántas consultas gana cada uno? Usa las cifras de tu propia tabla.
2. Explicación para no técnicos: uno de los dos modelos se entrenó para encontrar textos que "dicen lo mismo" y el otro para encontrar textos que responden a una pregunta. ¿Cuál necesita un buscador y por qué?
3. STS frente a recuperación: ¿por qué una buena nota en STS (similitud entre frases) no garantiza que un modelo sea un buen buscador de documentos?
4. Limitación: con 50 consultas, ¿qué diferencias te parecen ruido? Menciona también las anotaciones incompletas de FiQA y los distractores aleatorios. ¿Cómo lo mitigarías?
5. Conclusión prudente: ¿mantienes e5-small-v2? ¿Qué medirías antes de decidir?"""
print("✅ eval4_justificacion asignada.")

✅ eval4_justificacion asignada.

## Resumen de entregables

In [12]:
print(f"\nVariables de corrección:")
for var in ["eval1_estrategia", "eval2_recall", "eval3_busqueda", "eval4_justificacion"]:
    val = globals().get(var, None)
    status = "✅ definida" if val is not None else "⚠️ no definida"
    print(f"  {var}: {status}")


Variables de corrección:
  eval1_estrategia: ✅ definida
  eval2_recall: ✅ definida
  eval3_busqueda: ✅ definida
  eval4_justificacion: ✅ definida

# References